In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import os
pd.options.mode.chained_assignment = None
import optuna
import time
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold, train_test_split
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler, RobustScaler
import pickle
from collections import defaultdict
from sklearn.pipeline import Pipeline
import joblib
import json
import platform

In [3]:
path = "saved_files/dataset"
depths = ["in_0_1"]#, "in_1_2", "in_2_3", "in_3_4"]

dfs = {}
for archivo in os.listdir(path):
    if archivo.endswith("_features.csv") and any(depth in archivo for depth in depths):
        nombre_sin_extension = os.path.splitext(archivo)[0]  # sin .csv
        ruta_completa = os.path.join(path, archivo)
        # quitamos "_features" al final del nombre
        dfs[nombre_sin_extension[:-9]] = pd.read_csv(ruta_completa)



# Limpiamos valores nulos
for nombre_df, df in dfs.items():
    for band_set in ["rhow", "rhown","rtoa"]:
        dfs[nombre_df] = df.dropna()

dfs_to_keep = [
    'C2X-Complex_rhow_9x9_depth_in_0_1', 
    'TOA_15x15_depth_in_0_1',
    'C2X-Complex_rhown_9x9_depth_in_0_1',
    'C2X-Complex_rhow_5x5_depth_in_0_1',
    'C2RCC_rhow_5x5_depth_in_0_1',
    'C2RCC_rhow_15x15_depth_in_0_1', 
    'C2X-Complex_rhown_15x15_depth_in_0_1',
    'C2RCC_rhown_5x5_depth_in_0_1', 
    'C2X-Complex_rhow_15x15_depth_in_0_1',
    'C2RCC_rhow_9x9_depth_in_0_1',
    'C2X-Complex_rhow_5x5_depth_in_1_2', 
    'C2X_rhow_3x3_depth_in_1_2',
    'C2X-Complex_rhown_5x5_depth_in_1_2',
    'C2X-Complex_rhow_9x9_depth_in_1_2',
    'C2X-Complex_rhow_3x3_depth_in_1_2',
    'C2X-Complex_rhown_3x3_depth_in_1_2',
    'C2RCC_rhown_3x3_depth_in_1_2',
    'C2X-Complex_rhown_9x9_depth_in_1_2',
    'C2X-Complex_rhow_15x15_depth_in_1_2', 
    'C2X_rhow_5x5_depth_in_1_2',
    'TOA_15x15_depth_in_2_3',
    'TOA_9x9_depth_in_2_3',
    'TOA_5x5_depth_in_2_3', 
    'C2X-Complex_rhow_5x5_depth_in_2_3',
    'C2RCC_rhown_5x5_depth_in_2_3', 
    'TOA_3x3_depth_in_2_3',
    'C2X-Complex_rhown_5x5_depth_in_2_3', 
    'C2RCC_rhow_3x3_depth_in_2_3',
    'C2X-Complex_rhown_9x9_depth_in_2_3', 
    'C2X_rhow_9x9_depth_in_2_3',
    'TOA_9x9_depth_in_3_4',
    'TOA_3x3_depth_in_3_4',
    'TOA_5x5_depth_in_3_4',
    'C2X-Complex_rhow_5x5_depth_in_3_4', 
    'TOA_1x1_depth_in_3_4',
    'TOA_15x15_depth_in_3_4',
    'C2X-Complex_rhown_5x5_depth_in_3_4',
    'C2X-Complex_rhow_9x9_depth_in_3_4',
    'C2X-Complex_rhow_15x15_depth_in_3_4',
    'C2X-Complex_rhown_9x9_depth_in_3_4'
    ]

dfs = {k: dfs[k] for k in dfs_to_keep if k in dfs}


for nombre_df, df in dfs.items():
    # Marcamos las columnas de CHl alta (equivalente a quantile(0.93))
    df["High_Chl"] = df["Chl"]>5
    # Sacamos la estación de cada fecha
    df['Date'] = pd.to_datetime(df['Date'])
    def get_season(month):
        if month in [12, 1, 2]:
            return 'Invierno'
        elif month in [3, 4, 5]:
            return 'Primavera'
        elif month in [6, 7, 8]:
            return 'Verano'
        else:
            return 'Otoño'
    df['Season'] = df['Date'].dt.month.apply(get_season)
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype('category')
    df = pd.concat([df.drop(columns=["Season"]),pd.get_dummies(df["Season"])], axis=1)
    dfs[nombre_df] = df

/tmp/ipykernel_3949135/57408143.py:80: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include='object').columns:
/tmp/ipykernel_3949135/57408143.py:80: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes 

In [4]:
# Plantillas base con los parámetros que no se han optimizado con Optuna
base_params = {
    "XGB": {
        'device': 'cpu',
        'objective': 'reg:squarederror',
        'tree_method': 'hist',
        'enable_categorical': True,
        'eval_metric': 'rmse'
    },
    "LBM": {
        'objective': 'regression',
        'metric': 'rmse',
        'boosting_type': 'gbdt',
        'device': 'cpu',
        'verbosity': -1
    },
    "MLP": {
        'max_iter': 200,
        'shuffle': True,
        'tol': 1e-4,
        'n_iter_no_change': 25,
        'verbose': False,
        'early_stopping': True,
        'validation_fraction': 0.2
    },
    "SVR": {
        'kernel': 'rbf',
        'gamma': 'scale',
        'shrinking': True,
        'tol': 1e-3,
        'max_iter': -1,
        'verbose': False
    },
    "KNN": {
        'weights': 'distance',
        'algorithm': 'auto',
        'metric': 'minkowski',
        'p': 2,
        'n_jobs': -1
    },
    "RF": {
        'criterion': 'squared_error',
        'random_state': 42,
        'verbose': 0
    },
    "CAT": {
        'loss_function': 'RMSE',
        'eval_metric': 'RMSE',
        'random_seed': 42,
        'early_stopping_rounds': 50,
        'verbose': False
    },
    "ELN": {
        'fit_intercept': True,
        'max_iter': 1000,
        'tol': 1e-4,
        'selection': 'cyclic',
        'random_state': 42
    }
}

In [27]:
depth = "in_0_1"
seed = 1555

In [28]:
with open(f"training_results/seed{seed}/selection_results_{depth}.pkl", "rb") as f:
    results = pickle.load(f)

In [29]:
# Construir el diccionario final
model_params = {}
for (dataset, model), data in results.items():
    best_params = data["best_params"]
    # unimos base + best_params (best_params sobrescribe a base)
    merged = {**base_params.get(model, {}), **best_params}
    if dataset not in model_params:
        model_params[dataset] = {}
    model_params[dataset][model] = merged

In [30]:
def cross_validation_training(dfs, depth, seed):

    FOLDS = 5
    # Para hacer clip a las predicciones y forzar > 0
    correct = True
    results = {}
    rs = seed

    for nombre_df, df in list(dfs.items()):
    #for nombre_df, df in islice(dfs.items(), 3):
        print(f"\n=== Procesando {nombre_df} ===")
        # Ignoramos las columnas de Date, Lat, Lon y Buoy
        df = df.iloc[:, 4:]

        # Separamos el conjunto de datos en train y test: Train 75% Test 25%
        train, test = train_test_split(df, test_size=0.25, random_state=rs, stratify=df["High_Chl"])
        # Seleccionamos la columna que queremos predecir
        target = "Chl"

        # Quitamos esa columna y el indicador de clorofila alta
        X = train.drop(columns=[target, "High_Chl", "Turbidez"])
        # Para y cogemos solamente Chl
        y = train[target]
        # Para poder hacer StratifiedKFold y tener el mismo número de valores de Chl alta en cada fold
        y_class = train["High_Chl"]

        # Definimos X e y para test
        X_test = test.drop(columns=[target, "High_Chl", "Turbidez"])
        y_test = test[target]

        #----Definir modelos usando los params específicos de este dataset----# Esto es diferente a como están en Entrenamiento_V3
        models = {
            "XGB": XGBRegressor(**model_params[nombre_df]["XGB"]),
            "LBM": LGBMRegressor(**model_params[nombre_df]["LBM"]),
            "MLP": MLPRegressor(**model_params[nombre_df]["MLP"]),
            "SVR": SVR(**model_params[nombre_df]["SVR"]),
            "KNN": KNeighborsRegressor(**model_params[nombre_df]["KNN"]),
            "LR": LinearRegression(),
            "RF": RandomForestRegressor(**model_params[nombre_df]["RF"]),
            "CAT": CatBoostRegressor(**model_params[nombre_df]["CAT"]),
            "ELN": ElasticNet(**model_params[nombre_df]["ELN"])
        }


        # Dicts para guardar las predicciones sobre los conjuntos de validación, las y's correspondientes y los índices que corresponden dentro del loop de folds para el ensemble
        val_preds = {name: np.zeros(len(train)) for name in models}
        y_vals = defaultdict(list)
        val_indices = {}
        # Dict para guardar las predicciones sobre test
        test_preds = {name: np.zeros(len(test)) for name in models}
        
        # Dict para guardar resultados
        results[nombre_df] = {name: {'RMSE': [], 'R2': []} for name in models}
        # Stratified KFold de 5 folds
        skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=rs)

        # Loop para entrenar cada uno de los modelos
        for name, model in models.items():
            print(f"\n=== Training {name} ===")
            # Loop de folds, manteniendo la proporción de clases (Chl > 5) con y_class
            for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_class)):
                print(f"Fold {fold+1}")
                X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
                y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

                # Para modelos basados en distancias escalamos los datos
                if name in ["MLP", "SVR", "KNN", "LR", "ELN"]:
                    # Escalado dentro del loop de folds para evitar data leakage entre folds
                    scaler_X = RobustScaler()
                    scaler_y = RobustScaler()
                    X_train_scaled = scaler_X.fit_transform(X_train)
                    X_val_scaled = scaler_X.transform(X_val)
                    X_test_scaled = scaler_X.transform(X_test)
                    y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).ravel()
                    # Entrenamos modelo con datos escalados
                    model.fit(X_train_scaled, y_train_scaled)
                    # Predicción sobre val y test, haciendo la transformada inversa para devolver y a su escala
                    val_pred = scaler_y.inverse_transform(model.predict(X_val_scaled).reshape(-1, 1)).ravel()
                    test_pred = scaler_y.inverse_transform(model.predict(X_test_scaled).reshape(-1, 1)).ravel()
                # Para modelos basados en árboles no es necesario escalar
                else:
                    # Entrenamos el modelo
                    model.fit(X_train, y_train)
                    # Predicción sobre val y test
                    val_pred = model.predict(X_val)
                    test_pred = model.predict(X_test)

                if correct:
                    val_pred = np.clip(val_pred, 0.2, None)
                    test_pred = np.clip(test_pred, 0.2, None)

                # Guardamos las predicciones sobre val, las y's que les corresponden y los índices
                val_preds[name][val_idx] = val_pred
                if name == list(models.keys())[0]:
                    # Solo lo guardamos una vez
                    y_vals[fold] = y_val
                    val_indices[fold] = val_idx  # val_idx es un array de índices relativos a train

                # Guardamos la predicción de test, haciendo la media entre los folds
                test_preds[name] += test_pred / FOLDS

                # Calculamos y guardamos métricas
                rmse = np.sqrt(mean_squared_error(y_val, val_pred))
                r2 = r2_score(y_val, val_pred)
                results[nombre_df][name]['RMSE'].append(rmse)
                results[nombre_df][name]['R2'].append(r2)

        # Extendemos el dict de resultados con el ensemble
        results[nombre_df]["ENS"] = {'RMSE': [], 'R2': []}

        for fold in range(FOLDS):
            # Índices y valores del fold actual
            fold_val_idx = val_indices[fold]
            meta_X_val = np.vstack([val_preds[model][fold_val_idx] for model in models]).T
            meta_y_val = y_vals[fold]

            # Índices de entrenamiento: todos menos el fold actual
            train_folds = [i for i in range(FOLDS) if i != fold]
            train_idx = np.concatenate([val_indices[i] for i in train_folds])
            meta_X_train = np.vstack([val_preds[model][train_idx] for model in models]).T
            meta_y_train = y.iloc[train_idx]

            # Entrenamos el meta-modelo solo con los otros 4 folds
            meta_model = Ridge().fit(meta_X_train, meta_y_train)

            # Predicción en el fold actual (no visto)
            ensemble_pred = meta_model.predict(meta_X_val)
            if correct:
                ensemble_pred = np.clip(ensemble_pred, 0.2, None)

            rmse = np.sqrt(mean_squared_error(meta_y_val, ensemble_pred))
            r2 = r2_score(meta_y_val, ensemble_pred)
            results[nombre_df]["ENS"]['RMSE'].append(rmse)
            results[nombre_df]["ENS"]['R2'].append(r2)



    # === Evaluación final sobre test ===
        for name in models:
            rmse_test = np.sqrt(mean_squared_error(y_test, test_preds[name]))
            r2_test = r2_score(y_test, test_preds[name])
            results[nombre_df][name]["RMSE test"] = rmse_test
            results[nombre_df][name]["R2 test"] = r2_test

        # Construcción del meta-modelo sobre todo el conjunto de validación
        final_meta_X = np.vstack([val_preds[model] for model in models]).T
        final_meta_y = y.values
        ensemble_model = Ridge().fit(final_meta_X, final_meta_y)

        # Predicción sobre test del ensemble
        meta_X_test = np.vstack([test_preds[model] for model in models]).T
        ensemble_test_pred = ensemble_model.predict(meta_X_test)
        if correct:
            ensemble_test_pred = np.clip(ensemble_test_pred, 0.3, None)
        # Evaluación del ensemble sobre test
        rmse_ens_test = np.sqrt(mean_squared_error(y_test, ensemble_test_pred))
        r2_ens_test = r2_score(y_test, ensemble_test_pred)
        results[nombre_df]["ENS"]["RMSE test"] = rmse_ens_test
        results[nombre_df]["ENS"]["R2 test"] = r2_ens_test

        # === Guardar predicciones en CSV ===
        predictions_dir = f"training_results/seed{seed}/predictions"
        os.makedirs(predictions_dir, exist_ok=True)
        # Crear DataFrame con valores reales y predicciones de todos los modelos
        predictions_data = {"Real": y_test.values}
        for name in models:
            predictions_data[f"{name}_Predicted"] = test_preds[name]
        predictions_data["ENS_Predicted"] = ensemble_test_pred
        predictions_df = pd.DataFrame(predictions_data)
        predictions_csv_path = f"{predictions_dir}/{nombre_df}_predictions.csv"
        predictions_df.to_csv(predictions_csv_path, index=False)
        print(f"Predicciones guardadas en: {predictions_csv_path}")

    with open(f"training_results/seed{seed}/results_entrenamiento_final_{depth}.pkl", "wb") as f:
        pickle.dump(results, f)

    return results

In [8]:
depths[0]

'in_0_1'

In [31]:
results = cross_validation_training(dfs, depths[0], seed)


=== Procesando C2X-Complex_rhow_9x9_depth_in_0_1 ===

=== Training XGB ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training LBM ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training MLP ===
Fold 1
Fold 2
Fold 3


/home/antonio/.pyenv/versions/3.11.0/envs/clorofila/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Fold 4
Fold 5

=== Training SVR ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training KNN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training LR ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training RF ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training CAT ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training ELN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Predicciones guardadas en: training_results/seed1555/predictions/C2X-Complex_rhow_9x9_depth_in_0_1_predictions.csv

=== Procesando TOA_15x15_depth_in_0_1 ===

=== Training XGB ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training LBM ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training MLP ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training SVR ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training KNN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training LR ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training RF ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training CAT ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training ELN ===
Fold 1
Fold 

/home/antonio/.pyenv/versions/3.11.0/envs/clorofila/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.732e+02, tolerance: 2.770e-01
  model = cd_fast.enet_coordinate_descent(
/home/antonio/.pyenv/versions/3.11.0/envs/clorofila/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.549e+02, tolerance: 3.768e-01
  model = cd_fast.enet_coordinate_descent(
/home/antonio/.pyenv/versions/3.11.0/envs/clorofila/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of i

Fold 2
Fold 3
Fold 4
Fold 5

=== Training LBM ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training MLP ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training SVR ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training KNN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training LR ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training RF ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training CAT ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training ELN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Predicciones guardadas en: training_results/seed1555/predictions/C2X-Complex_rhown_9x9_depth_in_0_1_predictions.csv

=== Procesando C2X-Complex_rhow_5x5_depth_in_0_1 ===

=== Training XGB ===
Fold 1


...

/home/antonio/.pyenv/versions/3.11.0/envs/clorofila/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Fold 5


/home/antonio/.pyenv/versions/3.11.0/envs/clorofila/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(



=== Training SVR ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training KNN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training LR ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training RF ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training CAT ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training ELN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Predicciones guardadas en: training_results/seed1555/predictions/C2RCC_rhow_9x9_depth_in_0_1_predictions.csv


/home/antonio/.pyenv/versions/3.11.0/envs/clorofila/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.170e+02, tolerance: 3.986e-01
  model = cd_fast.enet_coordinate_descent(
/home/antonio/.pyenv/versions/3.11.0/envs/clorofila/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.715e+02, tolerance: 3.779e-01
  model = cd_fast.enet_coordinate_descent(
/home/antonio/.pyenv/versions/3.11.0/envs/clorofila/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of i

In [58]:
with open(f"training_results/seed{seed}/results_entrenamiento_final_{depth}.pkl", "rb") as f:
    results = pickle.load(f)

In [59]:
rows = []

for df_name, model_scores in results.items():
    row = {}
    for model_name, metrics in model_scores.items():
        for metric_name, values in metrics.items():
            if isinstance(values, list):  # Solo para los que tienen listas (folds)
                mean_val = np.mean(values)
                std_val = np.std(values)
                row[(metric_name, model_name)] = f"{mean_val:.2f} ± {std_val:.2f}"
            else:
                # Para el ensemble que tiene un único valor
                row[(metric_name, model_name)] = f"{values:.2f}"
    rows.append((df_name, row))

df_results = pd.DataFrame.from_dict(dict(rows), orient="index")
df_results.columns = pd.MultiIndex.from_tuples(df_results.columns, names=["Metric", "Model"])
df_results = df_results.sort_index(axis=1, level=0)
df_results = df_results.sort_index(axis=0)

In [60]:
df_sorted = df_results["R2 test"].copy()
# Añadir una columna auxiliar con el R2 máximo por fila
df_sorted["max_R2"] = df_sorted.max(axis=1)
# Ordenar por esa columna en orden descendente
df_sorted = df_sorted.sort_values("max_R2", ascending=False)
# Eliminar la columna auxiliar
df_sorted = df_sorted.drop(columns="max_R2")
df_sorted

Model,CAT,ELN,ENS,KNN,LBM,LR,MLP,RF,SVR,XGB
C2RCC_rhow_15x15_depth_in_0_1,0.78,0.53,0.79,0.81,0.75,0.60,0.76,0.67,0.58,0.71
TOA_15x15_depth_in_0_1,0.78,0.47,0.81,0.70,0.77,0.41,0.58,0.71,0.31,0.68
C2RCC_rhow_5x5_depth_in_0_1,0.79,0.48,0.68,0.67,0.80,0.53,0.80,0.67,0.58,0.72
C2RCC_rhow_9x9_depth_in_0_1,0.80,0.50,0.73,0.73,0.79,0.64,0.79,0.70,0.59,0.79
C2RCC_rhown_5x5_depth_in_0_1,0.77,0.49,0.70,0.71,0.78,0.46,0.73,0.66,0.64,0.71
C2X-Complex_rhown_15x15_depth_in_0_1,0.73,0.54,0.72,0.73,0.61,0.53,0.70,0.62,0.68,0.68
C2X-Complex_rhow_15x15_depth_in_0_1,0.71,0.57,0.68,0.72,0.58,0.59,0.65,0.58,0.65,0.58
C2X-Complex_rhown_9x9_depth_in_0_1,0.69,0.56,0.58,0.63,0.62,0.45,0.70,0.64,0.61,0.68
C2X-Complex_rhow_5x5_depth_in_0_1,0.68,0.59,0.68,0.61,0.61,0.60,0.62,0.58,0.63,0.61
C2X-Complex_rhow_9x9_depth_in_0_1,0.67,0.57,0.66,0.60,0.61,0.55,0.63,0.63,0.58,0.60


In [34]:
df_sorted = df_results["R2 test"].copy()
# Añadir una columna auxiliar con el R2 máximo por fila
df_sorted["max_R2"] = df_sorted.max(axis=1)
# Ordenar por esa columna en orden descendente
df_sorted = df_sorted.sort_values("max_R2", ascending=False)
# Eliminar la columna auxiliar
df_sorted = df_sorted.drop(columns="max_R2")
df_sorted

Model,CAT,ELN,ENS,KNN,LBM,LR,MLP,RF,SVR,XGB
TOA_15x15_depth_in_0_1,0.89,0.55,0.87,0.69,0.93,0.52,0.90,0.87,0.33,0.91
C2X-Complex_rhow_5x5_depth_in_0_1,0.75,0.50,0.84,0.74,0.72,0.49,0.71,0.58,0.76,0.67
C2X-Complex_rhown_15x15_depth_in_0_1,0.66,0.49,0.79,0.74,0.66,0.46,0.61,0.51,0.69,0.61
C2X-Complex_rhow_15x15_depth_in_0_1,0.73,0.51,0.77,0.78,0.62,0.45,0.66,0.55,0.75,0.62
C2RCC_rhow_15x15_depth_in_0_1,0.66,0.52,0.73,0.77,0.52,0.57,0.69,0.56,0.67,0.60
C2RCC_rhow_5x5_depth_in_0_1,0.68,0.52,0.65,0.74,0.69,0.55,0.73,0.62,0.67,0.63
C2RCC_rhown_5x5_depth_in_0_1,0.69,0.53,0.61,0.74,0.71,0.60,0.69,0.59,0.64,0.64
C2X-Complex_rhow_9x9_depth_in_0_1,0.68,0.51,0.69,0.70,0.59,0.52,0.65,0.57,0.74,0.60
C2X-Complex_rhown_9x9_depth_in_0_1,0.65,0.51,0.64,0.67,0.57,0.56,0.63,0.53,0.73,0.57
C2RCC_rhow_9x9_depth_in_0_1,0.64,0.53,0.67,0.69,0.55,0.51,0.64,0.57,0.65,0.59


In [27]:
df_sorted = df_results["R2 test"].copy()
# Añadir una columna auxiliar con el R2 máximo por fila
df_sorted["max_R2"] = df_sorted.max(axis=1)
# Ordenar por esa columna en orden descendente
df_sorted = df_sorted.sort_values("max_R2", ascending=False)
# Eliminar la columna auxiliar
df_sorted = df_sorted.drop(columns="max_R2")
df_sorted

Model,CAT,ELN,ENS,KNN,LBM,LR,MLP,RF,SVR,XGB
C2X-Complex_rhow_15x15_depth_in_0_1,0.84,0.70,0.81,0.80,0.78,0.67,0.74,0.78,0.88,0.80
C2X-Complex_rhow_9x9_depth_in_0_1,0.79,0.75,0.78,0.80,0.75,0.75,0.70,0.80,0.86,0.74
C2X-Complex_rhown_15x15_depth_in_0_1,0.84,0.64,0.84,0.85,0.75,0.71,0.72,0.77,0.85,0.79
C2X-Complex_rhown_9x9_depth_in_0_1,0.82,0.72,0.85,0.81,0.79,0.73,0.80,0.81,0.85,0.78
C2RCC_rhown_5x5_depth_in_0_1,0.80,0.63,0.75,0.80,0.84,0.66,0.73,0.79,0.78,0.83
C2X-Complex_rhow_5x5_depth_in_0_1,0.78,0.71,0.72,0.75,0.68,0.70,0.74,0.78,0.84,0.67
C2RCC_rhow_5x5_depth_in_0_1,0.81,0.63,0.80,0.80,0.82,0.66,0.76,0.77,0.79,0.83
C2RCC_rhow_9x9_depth_in_0_1,0.79,0.65,0.76,0.74,0.73,0.70,0.71,0.76,0.76,0.78
C2RCC_rhow_15x15_depth_in_0_1,0.76,0.38,0.74,0.71,0.72,0.28,0.64,0.74,0.74,0.75
TOA_15x15_depth_in_0_1,0.50,0.32,0.25,0.41,0.21,-0.02,-0.13,0.22,0.55,0.14


### Evaluación de resultados

In [152]:
def create_df_results(results, metric):
    rows = []
    for df_name, model_scores in results.items():
        row = {}
        for model_name, metrics in model_scores.items():
            for metric_name, values in metrics.items():
                if isinstance(values, list):  # Solo para los que tienen listas (folds)
                    mean_val = np.mean(values)
                    std_val = np.std(values)
                    row[(metric_name, model_name)] = f"{mean_val:.2f} ± {std_val:.2f}"
                else:
                    # Para el ensemble que tiene un único valor
                    row[(metric_name, model_name)] = f"{values:.2f}"
        rows.append((df_name, row))

    df_results = pd.DataFrame.from_dict(dict(rows), orient="index")
    df_results.columns = pd.MultiIndex.from_tuples(df_results.columns, names=["Metric", "Model"])
    df_results = df_results.sort_index(axis=1, level=0)
    df_results = df_results.sort_index(axis=0)

    if metric == "R2":
        df_sorted = df_results["R2 test"].copy()
        df_sorted["max_R2"] = df_sorted.max(axis=1)
        df_sorted = df_sorted.sort_values("max_R2", ascending=False)
        df_sorted_test = df_sorted.drop(columns="max_R2")

        df_sorted = df_results["R2"].copy()
        df_sorted["max_R2"] = df_sorted.max(axis=1)
        df_sorted = df_sorted.sort_values("max_R2", ascending=False)
        df_sorted_train = df_sorted.drop(columns="max_R2")

    elif metric == "RMSE":
        df_sorted = df_results["RMSE test"].copy()
        df_sorted["min_RMSE"] = df_sorted.min(axis=1)
        df_sorted = df_sorted.sort_values("min_RMSE", ascending=True)
        df_sorted_test = df_sorted.drop(columns="min_RMSE")

        df_sorted = df_results["RMSE"].copy()
        df_sorted["min_RMSE"] = df_sorted.min(axis=1)
        df_sorted = df_sorted.sort_values("min_RMSE", ascending=True)
        df_sorted_train = df_sorted.drop(columns="min_RMSE")

    return df_sorted_train, df_sorted_test

#### Profundidad 0-1

In [176]:
depth = "in_0_1"

with open(f"training_results/results_entrenamiento_final_{depth}.pkl", "rb") as f:
    results = pickle.load(f)

results_train, results_test = create_df_results(results, "R2")

results_train.to_csv(f"training_results/results_csv/results_{depth}_R2_train.csv")
results_test.to_csv(f"training_results/results_csv/results_{depth}_R2_test.csv")

In [174]:
results_train

Model,CAT,ELN,ENS,KNN,LBM,LR,MLP,RF,SVR,XGB
C2RCC_rhow_9x9_depth_in_0_1,0.76 ± 0.08,0.58 ± 0.09,0.71 ± 0.10,0.76 ± 0.14,0.63 ± 0.09,0.55 ± 0.10,0.59 ± 0.24,0.67 ± 0.08,0.72 ± 0.06,0.71 ± 0.08
C2RCC_rhow_5x5_depth_in_0_1,0.74 ± 0.08,0.59 ± 0.11,0.61 ± 0.15,0.74 ± 0.08,0.69 ± 0.14,0.55 ± 0.16,0.76 ± 0.05,0.67 ± 0.09,0.73 ± 0.04,0.71 ± 0.08
C2RCC_rhown_5x5_depth_in_0_1,0.75 ± 0.09,0.59 ± 0.12,0.64 ± 0.08,0.73 ± 0.08,0.73 ± 0.09,0.54 ± 0.21,0.66 ± 0.12,0.67 ± 0.05,0.73 ± 0.03,0.74 ± 0.08
C2X-Complex_rhow_15x15_depth_in_0_1,0.74 ± 0.13,0.58 ± 0.11,0.73 ± 0.09,0.70 ± 0.11,0.69 ± 0.12,0.05 ± 0.95,0.69 ± 0.17,0.63 ± 0.13,0.70 ± 0.14,0.69 ± 0.12
C2X-Complex_rhown_15x15_depth_in_0_1,0.74 ± 0.11,0.59 ± 0.12,0.63 ± 0.11,0.71 ± 0.10,0.67 ± 0.11,0.54 ± 0.16,0.61 ± 0.18,0.64 ± 0.13,0.68 ± 0.13,0.69 ± 0.12
TOA_15x15_depth_in_0_1,0.68 ± 0.15,0.49 ± 0.10,0.72 ± 0.10,0.73 ± 0.11,0.59 ± 0.11,0.30 ± 0.26,0.51 ± 0.14,0.67 ± 0.13,0.34 ± 0.03,0.65 ± 0.09
C2RCC_rhow_15x15_depth_in_0_1,0.70 ± 0.16,0.51 ± 0.11,0.61 ± 0.19,0.72 ± 0.08,0.57 ± 0.28,0.25 ± 0.81,0.67 ± 0.22,0.61 ± 0.22,0.68 ± 0.11,0.63 ± 0.20
C2X-Complex_rhow_9x9_depth_in_0_1,0.71 ± 0.09,0.48 ± 0.15,0.65 ± 0.10,0.61 ± 0.12,0.63 ± 0.10,0.40 ± 0.32,0.63 ± 0.13,0.60 ± 0.15,0.67 ± 0.10,0.66 ± 0.09
C2X-Complex_rhown_9x9_depth_in_0_1,0.69 ± 0.10,0.53 ± 0.11,0.65 ± 0.13,0.63 ± 0.13,0.52 ± 0.25,0.42 ± 0.27,0.67 ± 0.14,0.54 ± 0.19,0.68 ± 0.08,0.59 ± 0.18
C2X-Complex_rhow_5x5_depth_in_0_1,0.66 ± 0.11,0.49 ± 0.17,0.65 ± 0.14,0.66 ± 0.12,0.52 ± 0.17,0.38 ± 0.19,0.63 ± 0.13,0.55 ± 0.17,0.68 ± 0.08,0.52 ± 0.21


In [143]:
results_test

Model,CAT,ELN,ENS,KNN,LBM,LR,MLP,RF,SVR,XGB
C2X-Complex_rhow_9x9_depth_in_0_1,0.88,0.68,0.89,0.76,0.87,0.65,0.78,0.82,0.74,0.89
C2X-Complex_rhown_9x9_depth_in_0_1,0.86,0.70,0.86,0.78,0.85,0.65,0.82,0.80,0.74,0.84
C2RCC_rhow_5x5_depth_in_0_1,0.80,0.64,0.82,0.85,0.76,0.43,0.79,0.75,0.71,0.81
C2X-Complex_rhow_15x15_depth_in_0_1,0.82,0.68,0.84,0.85,0.81,0.68,0.78,0.79,0.72,0.81
TOA_15x15_depth_in_0_1,0.68,0.56,0.85,0.85,0.61,0.51,0.69,0.49,0.34,0.61
C2RCC_rhow_15x15_depth_in_0_1,0.77,0.58,0.84,0.84,0.71,0.61,0.77,0.76,0.70,0.78
C2RCC_rhown_5x5_depth_in_0_1,0.82,0.64,0.84,0.82,0.80,0.50,0.81,0.78,0.71,0.84
C2X-Complex_rhow_5x5_depth_in_0_1,0.84,0.53,0.76,0.77,0.84,-0.11,0.74,0.80,0.72,0.84
C2RCC_rhow_9x9_depth_in_0_1,0.81,0.67,0.80,0.79,0.80,0.65,0.74,0.76,0.72,0.83
C2X-Complex_rhown_15x15_depth_in_0_1,0.77,0.67,0.79,0.81,0.73,0.64,0.73,0.73,0.71,0.77


In [177]:
results_train, results_test = create_df_results(results, "RMSE")
results_train.to_csv(f"training_results/results_csv/results_{depth}_RMSE_train.csv")
results_test.to_csv(f"training_results/results_csv/results_{depth}_RMSE_test.csv")

In [154]:
results_test

Model,CAT,ELN,ENS,KNN,LBM,LR,MLP,RF,SVR,XGB
C2X-Complex_rhow_9x9_depth_in_0_1,1.51,2.45,1.43,2.10,1.58,2.56,2.00,1.82,2.18,1.41
C2X-Complex_rhown_9x9_depth_in_0_1,1.63,2.35,1.62,2.00,1.67,2.56,1.81,1.93,2.19,1.70
TOA_15x15_depth_in_0_1,2.42,2.82,1.63,1.66,2.67,2.99,2.39,3.06,3.48,2.68
C2RCC_rhow_5x5_depth_in_0_1,1.93,2.57,1.81,1.68,2.09,3.24,1.97,2.14,2.33,1.86
C2X-Complex_rhow_15x15_depth_in_0_1,1.80,2.42,1.71,1.68,1.85,2.43,2.02,1.96,2.28,1.86
C2RCC_rhown_5x5_depth_in_0_1,1.84,2.57,1.73,1.82,1.94,3.05,1.86,2.03,2.31,1.70
C2X-Complex_rhow_5x5_depth_in_0_1,1.73,2.95,2.09,2.05,1.70,4.53,2.20,1.94,2.26,1.72
C2RCC_rhow_15x15_depth_in_0_1,2.05,2.79,1.73,1.73,2.33,2.69,2.07,2.12,2.36,1.99
C2RCC_rhow_9x9_depth_in_0_1,1.88,2.47,1.92,1.98,1.93,2.54,2.19,2.08,2.29,1.79
C2X-Complex_rhown_15x15_depth_in_0_1,2.05,2.47,1.97,1.88,2.23,2.59,2.21,2.22,2.30,2.07


El conjunto que mejores resultados da es del de C2X-Complex_rhow_9x9_depth_in_0_1, con un R2 de 0.89 para XGB y también con el Ensemble. El RMSE es de 1.41 mientras que con el Ensemble es ligeramente mayor, de 1.43. CatBoost también da muy buenos resultados pero ligeramente inferiores a estos dos.

El segundo que mejores resultados da es el mismo conjunto pero con rhown. En líneas generales, vemos que los conjuntos que van con una mayor ventana de agregación funcionan mejor, y de hecho, en las 10 mejores no hay ninguno con agregación 1x1 o 3x3. Según el tipo de procesado, parece que C2X-Complex funciona mejor, seguido de C2RCC.

#### Profundidad 1-2

In [178]:
depth = "in_1_2"

with open(f"training_results/results_entrenamiento_final_{depth}.pkl", "rb") as f:
    results = pickle.load(f)

results_train, results_test = create_df_results(results, "R2")
results_train.to_csv(f"training_results/results_csv/results_{depth}_R2_train.csv")
results_test.to_csv(f"training_results/results_csv/results_{depth}_R2_test.csv")

In [157]:
results_train

Model,CAT,ELN,ENS,KNN,LBM,LR,MLP,RF,SVR,XGB
C2X-Complex_rhow_15x15_depth_in_1_2,0.68 ± 0.14,0.56 ± 0.24,0.69 ± 0.14,0.71 ± 0.14,0.59 ± 0.12,0.51 ± 0.23,0.69 ± 0.11,0.57 ± 0.19,0.75 ± 0.05,0.60 ± 0.18
C2X-Complex_rhown_9x9_depth_in_1_2,0.67 ± 0.05,0.59 ± 0.08,0.67 ± 0.08,0.65 ± 0.05,0.67 ± 0.11,0.36 ± 0.43,0.59 ± 0.16,0.64 ± 0.08,0.72 ± 0.06,0.68 ± 0.10
C2X-Complex_rhow_9x9_depth_in_1_2,0.69 ± 0.02,0.58 ± 0.08,0.67 ± 0.09,0.61 ± 0.06,0.65 ± 0.09,0.36 ± 0.32,0.59 ± 0.11,0.61 ± 0.09,0.71 ± 0.07,0.67 ± 0.09
C2RCC_rhown_3x3_depth_in_1_2,0.66 ± 0.06,0.55 ± 0.10,0.56 ± 0.14,0.67 ± 0.04,0.47 ± 0.27,0.35 ± 0.38,0.70 ± 0.06,0.55 ± 0.09,0.67 ± 0.15,0.57 ± 0.08
C2X-Complex_rhow_5x5_depth_in_1_2,0.63 ± 0.07,0.47 ± 0.22,0.59 ± 0.07,0.63 ± 0.12,0.60 ± 0.10,-0.06 ± 0.84,0.59 ± 0.04,0.51 ± 0.14,0.69 ± 0.07,0.59 ± 0.14
C2X-Complex_rhown_3x3_depth_in_1_2,0.67 ± 0.06,0.42 ± 0.29,0.57 ± 0.14,0.59 ± 0.13,0.55 ± 0.05,-0.57 ± 2.27,0.60 ± 0.15,0.56 ± 0.08,0.58 ± 0.21,0.64 ± 0.06
C2X-Complex_rhown_5x5_depth_in_1_2,0.61 ± 0.09,0.48 ± 0.21,0.54 ± 0.10,0.60 ± 0.12,0.51 ± 0.18,-0.61 ± 1.73,0.61 ± 0.08,0.51 ± 0.13,0.67 ± 0.06,0.57 ± 0.12
C2X-Complex_rhow_3x3_depth_in_1_2,0.66 ± 0.09,0.45 ± 0.23,0.56 ± 0.07,0.59 ± 0.12,0.60 ± 0.08,-0.23 ± 1.59,0.61 ± 0.08,0.61 ± 0.10,0.61 ± 0.15,0.63 ± 0.13
C2X_rhow_5x5_depth_in_1_2,0.52 ± 0.08,0.49 ± 0.18,0.43 ± 0.09,0.57 ± 0.10,0.39 ± 0.23,0.05 ± 0.84,0.54 ± 0.09,0.47 ± 0.08,0.59 ± 0.07,0.41 ± 0.10
C2X_rhow_3x3_depth_in_1_2,0.50 ± 0.10,0.51 ± 0.18,0.48 ± 0.16,0.52 ± 0.13,0.39 ± 0.09,0.30 ± 0.27,0.53 ± 0.10,0.49 ± 0.10,0.56 ± 0.10,0.46 ± 0.10


In [158]:
results_test

Model,CAT,ELN,ENS,KNN,LBM,LR,MLP,RF,SVR,XGB
C2X-Complex_rhow_5x5_depth_in_1_2,0.86,0.60,0.87,0.77,0.82,0.71,0.76,0.73,0.86,0.83
C2X-Complex_rhow_9x9_depth_in_1_2,0.86,0.55,0.84,0.76,0.80,0.53,0.66,0.79,0.83,0.83
C2X-Complex_rhown_5x5_depth_in_1_2,0.86,0.63,0.86,0.78,0.80,0.71,0.74,0.71,0.86,0.78
C2X_rhow_3x3_depth_in_1_2,0.82,0.71,0.62,0.75,0.84,0.67,0.75,0.77,0.63,0.83
C2X-Complex_rhow_3x3_depth_in_1_2,0.80,0.56,0.80,0.79,0.76,0.54,0.72,0.69,0.83,0.77
C2X-Complex_rhown_3x3_depth_in_1_2,0.82,0.59,0.83,0.75,0.74,0.67,0.74,0.70,0.78,0.77
C2X-Complex_rhown_9x9_depth_in_1_2,0.81,0.59,0.79,0.77,0.71,0.48,0.76,0.70,0.83,0.73
C2RCC_rhown_3x3_depth_in_1_2,0.81,0.60,0.81,0.81,0.76,0.62,0.79,0.58,0.78,0.76
C2X-Complex_rhow_15x15_depth_in_1_2,0.81,0.65,0.70,0.74,0.81,-0.44,0.75,0.80,0.77,0.78
C2X_rhow_5x5_depth_in_1_2,0.80,0.72,0.64,0.74,0.79,0.72,0.65,0.68,0.66,0.81


In [179]:
results_train, results_test = create_df_results(results, "RMSE")
results_train.to_csv(f"training_results/results_csv/results_{depth}_RMSE_train.csv")
results_test.to_csv(f"training_results/results_csv/results_{depth}_RMSE_test.csv")

In [160]:
results_test

Model,CAT,ELN,ENS,KNN,LBM,LR,MLP,RF,SVR,XGB
C2X-Complex_rhow_5x5_depth_in_1_2,1.60,2.70,1.53,2.05,1.82,2.28,2.09,2.20,1.62,1.75
C2X-Complex_rhown_5x5_depth_in_1_2,1.58,2.59,1.62,1.98,1.90,2.31,2.17,2.28,1.59,1.99
C2X-Complex_rhow_9x9_depth_in_1_2,1.61,2.85,1.70,2.10,1.92,2.91,2.50,1.97,1.75,1.75
C2X_rhow_3x3_depth_in_1_2,1.78,2.29,2.62,2.11,1.69,2.43,2.12,2.02,2.57,1.76
C2X-Complex_rhown_9x9_depth_in_1_2,1.85,2.72,1.94,2.05,2.28,3.06,2.08,2.33,1.75,2.22
C2X-Complex_rhow_3x3_depth_in_1_2,1.93,2.81,1.92,1.97,2.09,2.87,2.26,2.35,1.76,2.06
C2X-Complex_rhown_3x3_depth_in_1_2,1.81,2.72,1.76,2.11,2.18,2.45,2.18,2.31,1.98,2.05
C2RCC_rhown_3x3_depth_in_1_2,1.86,2.69,1.83,1.86,2.10,2.62,1.97,2.75,2.00,2.06
C2X-Complex_rhow_15x15_depth_in_1_2,1.84,2.52,2.34,2.15,1.84,5.11,2.12,1.89,2.05,2.01
C2X_rhow_5x5_depth_in_1_2,1.88,2.25,2.55,2.16,1.97,2.26,2.51,2.40,2.48,1.88


Aquí también hay una clara predominancia de los conjuntos procesados con C2X-Complex. Los mejores resultados vienen con C2X-Complex_rhow_5x5_depth_in_1_2, seguido del mismo con la ventana de 9x9, que es el mejor en la profundidad 0-1. En este caso es el Ensemble el modelo que mejores métricas da, con un R2 de 0.87 y un RMSE de 1.53, seguido de cerca por CAT, tanto en 5x5 como en 9X9.

#### Profundidad 2-3

In [180]:
depth = "in_2_3"

with open(f"training_results/results_entrenamiento_final_{depth}.pkl", "rb") as f:
    results = pickle.load(f)

results_train, results_test = create_df_results(results, "R2")
results_train.to_csv(f"training_results/results_csv/results_{depth}_R2_train.csv")
results_test.to_csv(f"training_results/results_csv/results_{depth}_R2_test.csv")

In [162]:
results_train

Model,CAT,ELN,ENS,KNN,LBM,LR,MLP,RF,SVR,XGB
TOA_9x9_depth_in_2_3,0.64 ± 0.09,0.44 ± 0.09,0.63 ± 0.11,0.69 ± 0.08,0.58 ± 0.14,0.37 ± 0.16,0.53 ± 0.19,0.57 ± 0.09,0.48 ± 0.04,0.59 ± 0.11
TOA_15x15_depth_in_2_3,0.65 ± 0.09,0.46 ± 0.07,0.66 ± 0.11,0.66 ± 0.10,0.50 ± 0.29,0.41 ± 0.13,0.50 ± 0.21,0.53 ± 0.09,0.48 ± 0.05,0.50 ± 0.20
C2X-Complex_rhown_5x5_depth_in_2_3,0.54 ± 0.26,0.49 ± 0.23,0.52 ± 0.29,0.60 ± 0.13,0.46 ± 0.26,0.22 ± 0.65,0.52 ± 0.22,0.50 ± 0.27,0.65 ± 0.07,0.51 ± 0.25
C2X-Complex_rhown_9x9_depth_in_2_3,0.49 ± 0.24,0.46 ± 0.24,0.49 ± 0.24,0.57 ± 0.15,0.47 ± 0.28,0.26 ± 0.65,0.49 ± 0.20,0.51 ± 0.23,0.64 ± 0.10,0.51 ± 0.23
TOA_5x5_depth_in_2_3,0.60 ± 0.10,0.47 ± 0.08,0.63 ± 0.12,0.60 ± 0.14,0.59 ± 0.12,0.43 ± 0.13,0.61 ± 0.07,0.46 ± 0.18,0.48 ± 0.06,0.57 ± 0.13
C2RCC_rhown_5x5_depth_in_2_3,0.60 ± 0.19,0.58 ± 0.13,0.53 ± 0.19,0.60 ± 0.20,0.59 ± 0.18,0.52 ± 0.18,0.63 ± 0.10,0.58 ± 0.19,0.61 ± 0.09,0.60 ± 0.15
C2X-Complex_rhow_5x5_depth_in_2_3,0.54 ± 0.24,0.48 ± 0.24,0.47 ± 0.40,0.60 ± 0.13,0.46 ± 0.31,0.20 ± 0.70,0.57 ± 0.12,0.51 ± 0.23,0.62 ± 0.12,0.51 ± 0.28
C2RCC_rhow_3x3_depth_in_2_3,0.55 ± 0.22,0.50 ± 0.17,0.55 ± 0.15,0.55 ± 0.21,0.45 ± 0.26,0.38 ± 0.22,0.57 ± 0.14,0.51 ± 0.26,0.62 ± 0.07,0.46 ± 0.30
TOA_3x3_depth_in_2_3,0.57 ± 0.12,0.46 ± 0.09,0.60 ± 0.14,0.58 ± 0.16,0.55 ± 0.11,0.37 ± 0.13,0.61 ± 0.10,0.50 ± 0.15,0.46 ± 0.06,0.53 ± 0.19
C2X_rhow_9x9_depth_in_2_3,0.54 ± 0.22,0.52 ± 0.25,0.48 ± 0.30,0.56 ± 0.14,0.44 ± 0.22,0.35 ± 0.39,0.60 ± 0.10,0.54 ± 0.16,0.51 ± 0.25,0.47 ± 0.28


In [163]:
results_test

Model,CAT,ELN,ENS,KNN,LBM,LR,MLP,RF,SVR,XGB
TOA_15x15_depth_in_2_3,0.73,0.40,0.81,0.78,0.65,0.21,0.63,0.62,0.52,0.66
TOA_9x9_depth_in_2_3,0.71,0.39,0.79,0.78,0.67,0.27,0.65,0.63,0.52,0.63
TOA_3x3_depth_in_2_3,0.74,0.43,0.76,0.71,0.72,0.35,0.72,0.65,0.48,0.69
TOA_5x5_depth_in_2_3,0.75,0.41,0.76,0.75,0.72,0.36,0.69,0.62,0.50,0.73
C2RCC_rhown_5x5_depth_in_2_3,0.71,0.58,0.73,0.70,0.67,0.61,0.69,0.64,0.65,0.68
C2X-Complex_rhown_9x9_depth_in_2_3,0.73,0.44,0.61,0.58,0.65,0.41,0.68,0.64,0.68,0.66
C2X-Complex_rhow_5x5_depth_in_2_3,0.72,0.16,0.71,0.59,0.63,0.24,0.64,0.55,0.72,0.65
C2X-Complex_rhown_5x5_depth_in_2_3,0.68,0.18,0.60,0.58,0.58,-0.48,0.68,0.54,0.72,0.61
C2RCC_rhow_3x3_depth_in_2_3,0.68,0.54,0.68,0.67,0.59,0.55,0.69,0.61,0.63,0.59
C2X_rhow_9x9_depth_in_2_3,0.69,0.62,0.62,0.58,0.64,0.64,0.63,0.63,0.67,0.65


In [181]:
results_train, results_test = create_df_results(results, "RMSE")
results_train.to_csv(f"training_results/results_csv/results_{depth}_RMSE_train.csv")
results_test.to_csv(f"training_results/results_csv/results_{depth}_RMSE_test.csv")

In [166]:
results_test

Model,CAT,ELN,ENS,KNN,LBM,LR,MLP,RF,SVR,XGB
TOA_15x15_depth_in_2_3,1.88,2.82,1.61,1.72,2.15,3.25,2.21,2.24,2.54,2.13
TOA_9x9_depth_in_2_3,1.97,2.86,1.67,1.70,2.10,3.12,2.16,2.23,2.53,2.23
TOA_5x5_depth_in_2_3,1.82,2.80,1.78,1.81,1.93,2.92,2.04,2.24,2.59,1.88
C2X-Complex_rhown_9x9_depth_in_2_3,1.81,2.61,2.16,2.25,2.06,2.67,1.97,2.09,1.96,2.03
TOA_3x3_depth_in_2_3,1.87,2.76,1.81,1.95,1.94,2.95,1.94,2.17,2.63,2.02
C2RCC_rhown_5x5_depth_in_2_3,1.86,2.27,1.82,1.91,2.00,2.17,1.95,2.08,2.06,1.97
C2X-Complex_rhow_5x5_depth_in_2_3,1.85,3.18,1.89,2.22,2.12,3.04,2.09,2.32,1.83,2.05
C2X-Complex_rhown_5x5_depth_in_2_3,1.96,3.16,2.19,2.26,2.26,4.23,1.98,2.36,1.84,2.18
C2RCC_rhow_3x3_depth_in_2_3,1.95,2.36,1.98,1.99,2.24,2.34,1.93,2.19,2.12,2.23
C2X_rhow_9x9_depth_in_2_3,1.93,2.14,2.13,2.26,2.09,2.08,2.10,2.12,2.01,2.05


Curiosamente, en esta profundidad son los conjuntos sin procesado, los TOA, los que claramente funcionan mejor, yendo además de ventanas mayor a menor (véase que es 15x15 el mejor, seguido de 9x9, 5x5 y 3x3). Aquí el mejor R2 viene del Ensemble con 0.81 y un RMSE de 1.61. Para esta profundidad, KNN también funciona de forma sobresaliente y será el que utilicemos en la práctica.

#### Profundidad 3-4

In [182]:
depth = "in_3_4"

with open(f"training_results/results_entrenamiento_final_{depth}.pkl", "rb") as f:
    results = pickle.load(f)

results_train, results_test = create_df_results(results, "R2")
results_train.to_csv(f"training_results/results_csv/results_{depth}_R2_train.csv")
results_test.to_csv(f"training_results/results_csv/results_{depth}_R2_test.csv")

In [169]:
results_train

Model,CAT,ELN,ENS,KNN,LBM,LR,MLP,RF,SVR,XGB
TOA_15x15_depth_in_3_4,0.55 ± 0.03,0.38 ± 0.16,0.46 ± 0.18,0.52 ± 0.14,0.47 ± 0.04,0.30 ± 0.18,0.56 ± 0.05,0.39 ± 0.16,0.53 ± 0.16,0.45 ± 0.06
TOA_9x9_depth_in_3_4,0.54 ± 0.03,0.39 ± 0.14,0.39 ± 0.26,0.51 ± 0.16,0.49 ± 0.05,0.38 ± 0.17,0.48 ± 0.13,0.42 ± 0.17,0.53 ± 0.17,0.51 ± 0.05
TOA_5x5_depth_in_3_4,0.53 ± 0.05,0.40 ± 0.18,0.41 ± 0.23,0.43 ± 0.14,0.45 ± 0.09,0.32 ± 0.25,0.40 ± 0.14,0.36 ± 0.21,0.52 ± 0.16,0.46 ± 0.10
C2X-Complex_rhow_15x15_depth_in_3_4,0.44 ± 0.20,0.35 ± 0.20,0.42 ± 0.34,0.50 ± 0.19,0.31 ± 0.27,0.12 ± 0.36,0.39 ± 0.17,0.40 ± 0.24,0.51 ± 0.14,0.39 ± 0.24
C2X-Complex_rhow_9x9_depth_in_3_4,0.45 ± 0.12,0.38 ± 0.21,0.38 ± 0.28,0.43 ± 0.21,0.33 ± 0.20,0.30 ± 0.17,0.21 ± 0.55,0.43 ± 0.17,0.51 ± 0.13,0.38 ± 0.19
TOA_1x1_depth_in_3_4,0.46 ± 0.08,0.39 ± 0.18,0.35 ± 0.30,0.42 ± 0.13,0.33 ± 0.15,0.39 ± 0.17,0.46 ± 0.05,0.32 ± 0.14,0.50 ± 0.17,0.38 ± 0.13
TOA_3x3_depth_in_3_4,0.43 ± 0.10,0.40 ± 0.15,0.34 ± 0.30,0.43 ± 0.10,0.35 ± 0.14,0.31 ± 0.18,0.47 ± 0.08,0.28 ± 0.19,0.50 ± 0.16,0.40 ± 0.10
C2X-Complex_rhow_5x5_depth_in_3_4,0.48 ± 0.18,0.37 ± 0.20,0.36 ± 0.30,0.40 ± 0.18,0.40 ± 0.15,0.26 ± 0.28,0.44 ± 0.24,0.40 ± 0.24,0.43 ± 0.14,0.41 ± 0.21
C2X-Complex_rhown_9x9_depth_in_3_4,0.38 ± 0.14,0.38 ± 0.12,0.40 ± 0.21,0.42 ± 0.20,0.38 ± 0.14,0.31 ± 0.22,0.43 ± 0.12,0.39 ± 0.20,0.46 ± 0.12,0.33 ± 0.15
C2X-Complex_rhown_5x5_depth_in_3_4,0.43 ± 0.15,0.37 ± 0.17,0.32 ± 0.22,0.42 ± 0.20,0.33 ± 0.37,0.30 ± 0.21,0.41 ± 0.18,0.38 ± 0.27,0.44 ± 0.14,0.35 ± 0.27


In [170]:
results_test

Model,CAT,ELN,ENS,KNN,LBM,LR,MLP,RF,SVR,XGB
TOA_15x15_depth_in_3_4,0.66,0.50,0.58,0.62,0.59,0.38,0.61,0.61,0.47,0.57
TOA_3x3_depth_in_3_4,0.59,0.46,0.55,0.61,0.64,0.40,0.60,0.66,0.45,0.61
C2X-Complex_rhow_5x5_depth_in_3_4,0.59,0.50,0.52,0.57,0.62,0.41,0.59,0.65,0.51,0.63
TOA_5x5_depth_in_3_4,0.58,0.44,0.50,0.59,0.64,0.45,0.58,0.63,0.42,0.62
C2X-Complex_rhown_5x5_depth_in_3_4,0.60,0.56,0.57,0.57,0.59,0.34,0.55,0.63,0.51,0.61
TOA_9x9_depth_in_3_4,0.61,0.46,0.63,0.62,0.59,0.38,0.62,0.61,0.47,0.58
C2X-Complex_rhow_15x15_depth_in_3_4,0.49,0.60,0.43,0.42,0.46,0.41,0.62,0.55,0.54,0.50
C2X-Complex_rhow_9x9_depth_in_3_4,0.55,0.56,0.51,0.55,0.57,0.46,0.61,0.62,0.58,0.60
TOA_1x1_depth_in_3_4,0.54,0.38,0.44,0.60,0.49,0.18,0.54,0.55,0.44,0.46
C2X-Complex_rhown_9x9_depth_in_3_4,0.59,0.58,0.50,0.52,0.54,0.37,0.57,0.59,0.55,0.58


In [183]:
results_train, results_test = create_df_results(results, "RMSE")
results_train.to_csv(f"training_results/results_csv/results_{depth}_RMSE_train.csv")
results_test.to_csv(f"training_results/results_csv/results_{depth}_RMSE_test.csv")

In [172]:
results_test

Model,CAT,ELN,ENS,KNN,LBM,LR,MLP,RF,SVR,XGB
C2X-Complex_rhow_5x5_depth_in_3_4,1.60,1.76,1.72,1.63,1.54,1.91,1.60,1.48,1.74,1.52
C2X-Complex_rhown_5x5_depth_in_3_4,1.58,1.67,1.64,1.64,1.60,2.02,1.68,1.52,1.75,1.56
C2X-Complex_rhow_9x9_depth_in_3_4,1.67,1.65,1.74,1.67,1.63,1.83,1.55,1.54,1.63,1.57
C2X-Complex_rhow_15x15_depth_in_3_4,1.80,1.60,1.91,1.92,1.86,1.95,1.56,1.69,1.72,1.80
C2X-Complex_rhown_9x9_depth_in_3_4,1.61,1.62,1.77,1.72,1.70,1.98,1.64,1.61,1.68,1.61
TOA_15x15_depth_in_3_4,1.74,2.10,1.93,1.83,1.91,2.34,1.86,1.86,2.16,1.94
TOA_3x3_depth_in_3_4,1.90,2.18,1.99,1.85,1.79,2.31,1.89,1.74,2.20,1.86
TOA_5x5_depth_in_3_4,1.92,2.23,2.10,1.91,1.78,2.21,1.93,1.81,2.27,1.84
TOA_9x9_depth_in_3_4,1.86,2.19,1.81,1.83,1.91,2.35,1.84,1.86,2.16,1.94
TOA_1x1_depth_in_3_4,2.01,2.35,2.22,1.87,2.13,2.70,2.02,2.00,2.22,2.19


En esta profundidad se nota un deterioro considerable de las métricas, bajando casi 0.2 en R2 respecto a la profunidad anterior. Los mejores modelos varían según R2 y RMSE, pero el que proporciona resultados buenos en ambas es C2X-Complex_rhow_5x5.



### Modelos finales

In [ ]:
path = "saved_files/dataset"
depths = ["in_0_1", "in_1_2", "in_2_3", "in_3_4"]

dfs = {}
for archivo in os.listdir(path):
    if archivo.endswith("_features.csv") and any(depth in archivo for depth in depths):
        nombre_sin_extension = os.path.splitext(archivo)[0]  # sin .csv
        ruta_completa = os.path.join(path, archivo)
        # quitamos "_features" al final del nombre
        dfs[nombre_sin_extension[:-9]] = pd.read_csv(ruta_completa)



# Limpiamos valores nulos
for nombre_df, df in dfs.items():
    for band_set in ["rhow", "rhown","rtoa"]:
        dfs[nombre_df] = df.dropna()

dfs_to_keep = [
    'C2X-Complex_rhow_15x15_depth_in_0_1', 
    'C2X-Complex_rhow_15x15_depth_in_1_2',
    'TOA_15x15_depth_in_2_3',
    'TOA_15x15_depth_in_3_4'
    ]

dfs = {k: dfs[k] for k in dfs_to_keep if k in dfs}


for nombre_df, df in dfs.items():
    # Marcamos las columnas de CHl alta (equivalente a quantile(0.93))
    threshold = df["Chl"].quantile(0.9)
    df["High_Chl"] = df["Chl"] > threshold
    
    # Sacamos la estación de cada fecha
    df['Date'] = pd.to_datetime(df['Date'])
    def get_season(month):
        if month in [12, 1, 2]:
            return 'Invierno'
        elif month in [3, 4, 5]:
            return 'Primavera'
        elif month in [6, 7, 8]:
            return 'Verano'
        else:
            return 'Otoño'
            
    df['Season'] = df['Date'].dt.month.apply(get_season)
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype('category')
    df = pd.concat([df.drop(columns=["Season"]),pd.get_dummies(df["Season"])], axis=1)
    dfs[nombre_df] = df

/tmp/ipykernel_624966/3307236790.py:46: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include='object').columns:
/tmp/ipykernel_624966/3307236790.py:46: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtype

In [3]:
def load_and_prepare_datasets(depth, path="saved_files/dataset"):
    """
    Carga y prepara dataframes para una profundidad específica.
    
    Args:
        depth: Profundidad (ej: "in_0_1", "in_1_2")
        datasets_to_keep: Lista de nombres de datasets a mantener (opcional) - Puesto a mano dentro de la función
        path: Ruta al directorio con los archivos CSV
    
    Returns:
        dict: Diccionario {nombre_dataset: dataframe_preparado}
    """
    # Cargamos los csv de los tifs
    dfs = {}
    for archivo in os.listdir(path):
        if archivo.endswith(f"{depth}_features.csv"):
            nombre_sin_extension = os.path.splitext(archivo)[0]  # sin .csv
            ruta_completa = os.path.join(path, archivo)
            dfs[nombre_sin_extension[:-9]] = pd.read_csv(ruta_completa)
    
    # Limpiamos valores nulos
    for nombre_df, df in dfs.items():
        for band_set in ["rhow", "rhown", "rtoa"]:
            dfs[nombre_df] = df.dropna()
    
    # Filtrar datasets si se especifica

    datasets_to_keep = [
        'C2X-Complex_rhow_15x15_depth_in_0_1', 
        'C2X-Complex_rhow_15x15_depth_in_1_2',
        'TOA_15x15_depth_in_2_3',
        'TOA_15x15_depth_in_3_4'
        ]

    if datasets_to_keep is not None:
        dfs = {k: dfs[k] for k in datasets_to_keep if k in dfs}
    
    # Procesamiento de cada dataframe
    for nombre_df, df in dfs.items():
        # Marcamos las columnas de Chl alta (equivalente a quantile(0.9))
        threshold = df["Chl"].quantile(0.9)
        df["High_Chl"] = df["Chl"] > threshold
        
        # Sacamos la estación de cada fecha
        df['Date'] = pd.to_datetime(df['Date'])
        
        def get_season(month):
            if month in [12, 1, 2]:
                return 'Invierno'
            elif month in [3, 4, 5]:
                return 'Primavera'
            elif month in [6, 7, 8]:
                return 'Verano'
            else:
                return 'Otoño'
        
        df['Season'] = df['Date'].dt.month.apply(get_season)
        
        # Convertir a categorías y hacer one-hot encoding
        for col in df.select_dtypes(include=['object', 'string']).columns:
            df[col] = df[col].astype('category')
        
        df = pd.concat([df.drop(columns=["Season"]), pd.get_dummies(df["Season"])], axis=1)
        dfs[nombre_df] = df
    
    return dfs

In [ ]:
dfs = load_and_prepare_datasets("") # argumento de depth vacío para que los coja todos

In [8]:
"""
=============================================================================
MODELOS FINALES - Entrenamiento con Optuna sobre dataset completo
=============================================================================

Flujo:
  1. Optuna busca los mejores hiperparámetros de CAT usando validación cruzada
     sobre el dataset COMPLETO (sin split train/test).
  2. Con los mejores parámetros encontrados, se entrena el modelo final
     sobre todos los datos.
  3. Se guarda el modelo, las features y los metadatos.

Datasets seleccionados (uno por profundidad):
  - in_0_1 -> C2X-Complex_rhow_15x15_depth_in_0_1 + CAT
  - in_1_2 -> C2X-Complex_rhow_15x15_depth_in_1_2 + CAT
  - in_2_3 -> TOA_15x15_depth_in_2_3              + CAT
  - in_3_4 -> TOA_15x15_depth_in_3_4              + CAT
=============================================================================
"""

import os
import json
import platform
import numpy as np
import pandas as pd
import optuna
import joblib
import pickle

from catboost import CatBoostRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import r2_score, mean_squared_log_error

optuna.logging.set_verbosity(optuna.logging.WARNING)

# ─────────────────────────────────────────────
# CONFIGURACIÓN
# ─────────────────────────────────────────────
TARGET        = "Chl"
FOLDS         = 5          # Número de folds para CV
N_TRIALS      = 100        # Trials de Optuna
SEED          = 42
SAVE_PATH     = "training_results/models"
HYPERPARAMS   = "hyperparams.json"

os.makedirs(SAVE_PATH, exist_ok=True)

# Datasets seleccionados (resultado del análisis multi-seed)
SELECTED = {
    "C2X-Complex_rhow_15x15_depth_in_0_1": dfs["C2X-Complex_rhow_15x15_depth_in_0_1"],
    "C2X-Complex_rhow_15x15_depth_in_1_2": dfs["C2X-Complex_rhow_15x15_depth_in_1_2"],
    "TOA_15x15_depth_in_2_3":              dfs["TOA_15x15_depth_in_2_3"],
    "TOA_15x15_depth_in_3_4":              dfs["TOA_15x15_depth_in_3_4"],
}

# ─────────────────────────────────────────────
# FUNCIÓN OBJETIVO OPTUNA (CV sobre dataset completo)
# ─────────────────────────────────────────────
def objective_cat(trial, df, seed):
    """
    Función objetivo para CAT.
    Usa validación cruzada estratificada sobre el dataset completo.
    Retorna (rmsle_mean, r2_mean) para optimización multi-objetivo.
    """
    # Cargar espacio de búsqueda desde hyperparams.json
    with open(HYPERPARAMS, "r") as f:
        config = json.load(f)["CAT"]
    param_config = config["params"]

    # Construir parámetros del trial
    params = {}
    for param_name, param_spec in param_config.items():
        if isinstance(param_spec, dict) and "type" in param_spec:
            ptype = param_spec["type"]
            if ptype == "categorical":
                params[param_name] = trial.suggest_categorical(param_name, param_spec["choices"])
            elif ptype == "int":
                params[param_name] = trial.suggest_int(param_name, param_spec["low"], param_spec["high"])
            elif ptype == "float":
                params[param_name] = trial.suggest_float(
                    param_name, param_spec["low"], param_spec["high"],
                    log=param_spec.get("log", False)
                )
        else:
            # Parámetro fijo
            params[param_name] = param_spec

    # Preparar X e y (ignorar Date, Lat, Lon, Buoy)
    df_work = df.iloc[:, 4:].copy()
    X = df_work.drop(columns=[TARGET, "High_Chl", "Turbidez"])
    y = df_work[TARGET]
    y_class = df_work["High_Chl"]   # Para StratifiedKFold

    rmsle_folds, r2_folds = [], []
    skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=seed)

    for train_idx, val_idx in skf.split(X, y_class):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model = CatBoostRegressor(**params)
        model.fit(X_train, y_train, eval_set=(X_val, y_val), verbose=False)

        val_pred = model.predict(X_val)
        val_pred = np.clip(val_pred, 0.2, None)   # Forzar predicciones > 0

        rmsle_folds.append(np.sqrt(mean_squared_log_error(y_val, val_pred)))
        r2_folds.append(r2_score(y_val, val_pred))

    return np.mean(rmsle_folds), np.mean(r2_folds)


# ─────────────────────────────────────────────
# FUNCIÓN PRINCIPAL
# ─────────────────────────────────────────────
def run_final_cat(selected_dfs, n_trials=N_TRIALS, seed=SEED):
    """
    Para cada dataset seleccionado:
      1. Ejecuta Optuna (multi-objetivo: minimizar RMSLE, maximizar R2)
         sobre el dataset completo con CV.
      2. Selecciona el mejor trial del frente de Pareto.
      3. Entrena el modelo final con todos los datos.
      4. Guarda modelo, features y metadatos.
    """
    all_results = {}

    for nombre_df, df in selected_dfs.items():
        print(f"\n{'='*65}")
        print(f"Optimizando CAT para: {nombre_df}")
        print(f"   Dataset: {len(df)} muestras | {df.shape[1]} columnas")
        print(f"   Trials Optuna: {n_trials} | Folds CV: {FOLDS}")
        print(f"{'='*65}")

        # ── 1. Optuna multi-objetivo ──────────────────────────────────
        study = optuna.create_study(
            directions=["minimize", "maximize"],   # RMSLE min, R2 max
            sampler=optuna.samplers.TPESampler(
                constant_liar=True,
                n_startup_trials=10,
                seed=seed
            )
        )
        study.optimize(
            lambda trial: objective_cat(trial, df, seed),
            n_trials=n_trials,
            n_jobs=1,          # CatBoost ya usa múltiples threads internamente
            show_progress_bar=True
        )

        # ── 2. Selección del mejor trial (frente de Pareto) ──────────
        pareto_trials = study.best_trials
        if not pareto_trials:
            pareto_trials = study.trials

        # Criterio de selección: score combinado
        # 50% RMSLE normalizado + 35% R2 normalizado (invertido) + 15% estabilidad
        rmsle_vals = np.array([t.values[0] for t in pareto_trials])
        r2_vals    = np.array([t.values[1] for t in pareto_trials])

        if len(pareto_trials) > 1:
            rmsle_norm = (rmsle_vals - rmsle_vals.min()) / (rmsle_vals.max() - rmsle_vals.min() + 1e-9)
            r2_norm    = 1 - (r2_vals - r2_vals.min()) / (r2_vals.max() - r2_vals.min() + 1e-9)
            score      = 0.65 * rmsle_norm + 0.35 * r2_norm
            best_trial = pareto_trials[int(np.argmin(score))]
        else:
            best_trial = pareto_trials[0]

        best_params  = best_trial.params
        best_rmsle   = best_trial.values[0]
        best_r2      = best_trial.values[1]

        print(f"\nMejor trial encontrado:")
        print(f"   RMSLE CV: {best_rmsle:.4f}")
        print(f"   R2 CV:    {best_r2:.4f}")
        print(f"   Params:   {best_params}")

        # ── 3. Parámetros finales: fijos del JSON + optimizados ───────
        with open(HYPERPARAMS, "r") as f:
            config = json.load(f)["CAT"]

        final_params = {}
        for param_name, param_spec in config["params"].items():
            if not isinstance(param_spec, dict) or "type" not in param_spec:
                final_params[param_name] = param_spec   # parámetros fijos
        final_params.update(best_params)                # sobrescribir con optimizados

        # ── 4. Entrenamiento final sobre TODOS los datos ──────────────
        df_work = df.iloc[:, 4:].copy()
        X_full  = df_work.drop(columns=[TARGET, "High_Chl", "Turbidez"])
        y_full  = df_work[TARGET]
        feature_names = X_full.columns.tolist()

        print(f"\nEntrenando modelo final sobre {len(X_full)} muestras...")
        final_model = CatBoostRegressor(**final_params)
        final_model.fit(X_full, y_full, verbose=False)
        print(f"   Modelo entrenado")

        # ── 5. Guardar artefactos ─────────────────────────────────────
        model_name = "CAT"
        base_name  = f"{nombre_df}_{model_name}"

        # 5a. Modelo joblib
        joblib_path = os.path.join(SAVE_PATH, f"{base_name}_model.joblib")
        joblib.dump(final_model, joblib_path)

        # 5b. Modelo nativo CatBoost (.cbm)
        cbm_path = os.path.join(SAVE_PATH, f"{base_name}_model.cbm")
        final_model.save_model(cbm_path)

        # 5c. Features
        features_path = os.path.join(SAVE_PATH, f"{base_name}_features.json")
        with open(features_path, "w") as f:
            json.dump({"feature_names": feature_names}, f, indent=2)

        # 5d. Metadatos
        import catboost, sklearn, numpy, pandas as pd_meta
        meta = {
            "dataset":      nombre_df,
            "model_name":   model_name,
            "best_params":  final_params,
            "cv_rmsle":     round(best_rmsle, 4),
            "cv_r2":        round(best_r2, 4),
            "n_trials":     n_trials,
            "n_folds":      FOLDS,
            "n_samples":    len(X_full),
            "n_features":   len(feature_names),
            "python":       platform.python_version(),
            "libs": {
                "catboost": catboost.__version__,
                "sklearn":  sklearn.__version__,
                "numpy":    numpy.__version__,
                "pandas":   pd_meta.__version__,
            }
        }
        meta_path = os.path.join(SAVE_PATH, f"{base_name}_metadata.json")
        with open(meta_path, "w") as f:
            json.dump(meta, f, indent=2)

        print(f"   Guardado en: {SAVE_PATH}/")
        print(f"      - {base_name}_model.joblib")
        print(f"      - {base_name}_model.cbm")
        print(f"      - {base_name}_features.json")
        print(f"      - {base_name}_metadata.json")

        all_results[nombre_df] = {
            "model":       final_model,
            "best_params": final_params,
            "cv_rmsle":    best_rmsle,
            "cv_r2":       best_r2,
        }

    print(f"\n{'='*65}")
    print("ENTRENAMIENTO FINAL COMPLETADO")
    print(f"{'='*65}\n")

    # Resumen
    print(f"{'Dataset':<45} {'RMSLE CV':>10} {'R2 CV':>8}")
    print("-" * 65)
    for nombre_df, res in all_results.items():
        print(f"{nombre_df:<45} {res['cv_rmsle']:>10.4f} {res['cv_r2']:>8.4f}")

    return all_results



In [9]:

# ─────────────────────────────────────────────
# EJECUCIÓN
# ─────────────────────────────────────────────
final_results = run_final_cat(SELECTED, n_trials=N_TRIALS, seed=SEED)


Optimizando CAT para: C2X-Complex_rhow_15x15_depth_in_0_1
   Dataset: 487 muestras | 63 columnas
   Trials Optuna: 100 | Folds CV: 5


/home/antonio/.pyenv/versions/3.11.0/envs/clorofila/lib/python3.11/site-packages/optuna/_experimental.py:33: ExperimentalWarning: Argument ``constant_liar`` is an experimental feature. The interface can change in the future.
  optuna_warn(


  0%|          | 0/100 [00:00<?, ?it/s]


Mejor trial encontrado:
   RMSLE CV: 0.3469
   R2 CV:    0.7690
   Params:   {'iterations': 500, 'learning_rate': 0.05, 'depth': 8, 'l2_leaf_reg': 2.5306373719089135}

Entrenando modelo final sobre 487 muestras...
   Modelo entrenado
   Guardado en: training_results/models/
      - C2X-Complex_rhow_15x15_depth_in_0_1_CAT_model.joblib
      - C2X-Complex_rhow_15x15_depth_in_0_1_CAT_model.cbm
      - C2X-Complex_rhow_15x15_depth_in_0_1_CAT_features.json
      - C2X-Complex_rhow_15x15_depth_in_0_1_CAT_metadata.json

Optimizando CAT para: C2X-Complex_rhow_15x15_depth_in_1_2
   Dataset: 475 muestras | 63 columnas
   Trials Optuna: 100 | Folds CV: 5


/home/antonio/.pyenv/versions/3.11.0/envs/clorofila/lib/python3.11/site-packages/optuna/_experimental.py:33: ExperimentalWarning: Argument ``constant_liar`` is an experimental feature. The interface can change in the future.
  optuna_warn(


...


Mejor trial encontrado:
   RMSLE CV: 0.3561
   R2 CV:    0.7949
   Params:   {'iterations': 500, 'learning_rate': 0.05, 'depth': 7, 'l2_leaf_reg': 1.1064493925754044}

Entrenando modelo final sobre 475 muestras...
   Modelo entrenado
   Guardado en: training_results/models/
      - C2X-Complex_rhow_15x15_depth_in_1_2_CAT_model.joblib
      - C2X-Complex_rhow_15x15_depth_in_1_2_CAT_model.cbm
      - C2X-Complex_rhow_15x15_depth_in_1_2_CAT_features.json
      - C2X-Complex_rhow_15x15_depth_in_1_2_CAT_metadata.json

Optimizando CAT para: TOA_15x15_depth_in_2_3
   Dataset: 477 muestras | 68 columnas
   Trials Optuna: 100 | Folds CV: 5


/home/antonio/.pyenv/versions/3.11.0/envs/clorofila/lib/python3.11/site-packages/optuna/_experimental.py:33: ExperimentalWarning: Argument ``constant_liar`` is an experimental feature. The interface can change in the future.
  optuna_warn(


  0%|          | 0/100 [00:00<?, ?it/s]

[W 2026-02-18 14:02:54,786] Trial 9 failed with parameters: {'iterations': 500, 'learning_rate': 0.04, 'depth': 8, 'l2_leaf_reg': 2.888859700647797} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/antonio/.pyenv/versions/3.11.0/envs/clorofila/lib/python3.11/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_624966/2337596280.py", line 144, in <lambda>
    lambda trial: objective_cat(trial, df, seed),
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_624966/2337596280.py", line 102, in objective_cat
    model.fit(X_train, y_train, eval_set=(X_val, y_val), verbose=False)
  File "/home/antonio/.pyenv/versions/3.11.0/envs/clorofila/lib/python3.11/site-packages/catboost/core.py", line 5873, in fit
    return self._fit(X, y, cat_features, text_features, embedding_features, None, graph, sample_weight, None, None, 

KeyboardInterrupt: 